In [1]:
import os
import gc
import json
import math
import pickle
import subprocess
import collections
import unicodedata

In [2]:
!pip install transformers

In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import OrderedDict
from torch.utils.data import TensorDataset, DataLoader
import tqdm.auto as tqdm
%matplotlib inline

from transformers import AutoModel, AutoTokenizer

/data/home/samsadalam/training-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DIRECTORY = '/data/home/samsadalam/MLSP_Project'

In [5]:
# these 2 checkpoints were chosen as they have vocab
INP_CP_NAME ='sentence-transformers/all-MiniLM-L6-v2'
OUT_CP_NAME = 'google/electra-base-generator'

In [6]:
!pip install 'accelerate>=0.26.0'

In [35]:
model_inp = AutoModel.from_pretrained(INP_CP_NAME, torch_dtype="auto", device_map="cuda:2")
model_out = AutoModel.from_pretrained(OUT_CP_NAME, torch_dtype="auto", device_map="cuda:2")
tokenizer_inp = AutoTokenizer.from_pretrained(INP_CP_NAME)
tokenizer_out = AutoTokenizer.from_pretrained(OUT_CP_NAME)

# Creating dataset

In [36]:
class Embed_dataset(TensorDataset):
    def __init__(self, tokenizer, inp_model, out_model):
        self.tokenizer = tokenizer
        self.inp_embed = inp_model.get_input_embeddings().weight.clone().detach()
        self.out_embed = out_model.get_input_embeddings().weight.clone().detach()
    def __len__(self):
        return self.tokenizer.vocab_size
    def __getitem__(self, idx):
        return self.inp_embed[idx], self.out_embed[idx]

In [37]:
full_dataset = Embed_dataset(tokenizer_inp, model_inp, model_out)
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [0.8, 0.2]) # WOW!!

In [38]:
INP_DIM = train_dataset[0][0].shape[0]
OUT_DIM = train_dataset[0][1].shape[0]

## Adapter model

In [39]:
class Hidden_block(torch.nn.Module):
    def __init__(self, hid_dim, num_layers, activation = 'relu', dropout = 0.1):
        super().__init__()
        self.layers = [(torch.nn.Linear(hid_dim, hid_dim), torch.nn.BatchNorm1d(hid_dim)) for _ in range(num_layers)]
        self.activation = torch.nn.functional.relu if activation == 'relu' else torch.nn.functional.tanh
        self.p = dropout
    def _apply(self, fn):
        for l, b in self.layers:
            l._apply(fn)
            b._apply(fn)

    def forward(self, inp):
        residual = inp
        for i in range(len(self.layers)):
            inp = self.layers[i][0](inp) # linear
            inp = self.activation(inp)
            inp = self.layers[i][1](inp) # batchnorm
            inp = torch.nn.functional.dropout(inp, p = self.p)
        return residual + inp

class Embed_Adapter(torch.nn.Module):
    def __init__(self, inp_dim, hid_dim, out_dim, activation = 'relu', init_num_blocks = 1, layers_per_block = 3, dropout = 0.1):
        super().__init__()
        self._dummy_param = torch.nn.Parameter(torch.tensor(0), requires_grad=False)
        self.inp_dim = inp_dim
        self.hid_dim = hid_dim
        self.out_dim = out_dim
        self.activation = activation
        self.layers_per_block = layers_per_block
        self.num_blocks = 0
        self.dropout = dropout
        self.layers = torch.nn.Sequential(OrderedDict([
            ('inp_layer', torch.nn.Linear(inp_dim, hid_dim)),
            (activation + '1', torch.nn.ReLU() if activation == 'relu' else torch.nn.Tanh()),
            ('dropout1', torch.nn.Dropout(dropout))
        ]))
        self.out_layer = torch.nn.Linear(hid_dim, out_dim)
        for _ in range(init_num_blocks):
            self.add_block()
    @property
    def device(self):
        return self._dummy_param.device

    def add_block(self):
        self.layers.append(Hidden_block(self.hid_dim, self.layers_per_block, self.activation, self.dropout))
        self.num_blocks += 1
    def forward(self, inp):
        inp = inp.to(self.device)
        inp = self.layers(inp)
        return self.out_layer(inp)

In [40]:
# Sanity testing
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
for x, y in train_dataloader:
    break
model = Embed_Adapter(inp_dim = INP_DIM, hid_dim = (INP_DIM + OUT_DIM)//2, out_dim = OUT_DIM, activation = 'relu', init_num_blocks = 3, layers_per_block = 3, dropout = 0.1)
y_pred = model.forward(x)
print(y.shape)
print(y_pred.shape)
y_batch = y.to(model.device)

torch.Size([64, 768])
torch.Size([64, 768])


In [41]:
(torch.nn.functional.normalize(y_pred, dim = 1)*torch.nn.functional.normalize(y_batch, dim = 1)).shape

torch.Size([64, 768])

In [42]:
torch.mean(torch.sum(torch.nn.functional.normalize(y_pred, dim = 1)*torch.nn.functional.normalize(y_pred, dim = 1), dim = 1))


tensor(1., grad_fn=<MeanBackward0>)

## Model agnostic trainer

In [43]:
def sync_vram():
    """ Synchronizes the VRAM across the GPUs, reclaiming unused memory. """
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [44]:
## ==== BEGIN EVALUATION PORTION

class Trainer:
    """ Performs model training in a model-agnostic manner.
        Requires specifying the model instance, the loss criterion to optimize,
          the optimizer to use and the directory to save data to.
    """

    def __init__(self, directory, model, criterion, optimizer, force_cpu = False):
        """ Initializes the trainer.

        Args:
            directory (str): Directory to save checkpoints and the model data in.
            model (torch.nn.Module): Torch model (must inherit `torch.nn.Module`) to train.
            criterion (torch.nn.Function): Loss criterion, i.e., the loss function to optimize for training.
            optimizer (torch.optim.Optimizer): Optimizer to use for training.
        """

        self.model            = model
        self.optimizer        = optimizer
        self.criterion        = criterion
        self.directory        = directory
        self.last_checkpoint  = 0
        self.loss_history     = { 'train': [], 'valid': [], 'train_cosine_sim': [], 'val_cosine_sim': [] }

        os.makedirs(self.directory, exist_ok=True)
        self.device = 'cuda:2' if (torch.cuda.is_available() and not force_cpu) else 'cpu'

    @staticmethod
    def make_dataloader(dataset, shuffle_data=True, batch_size=8, collate_fn=None):
        """ Create a dataloader for a torch Dataset.

        Args:
            dataset (torch.utils.data.Dataset): Dataset to process.
            shuffle_data (bool, optional): If true, shuffles the data. Defaults to True.
            batch_size (int, optional): Number of items per batch. Defaults to 8.
            collate_fn (function, optional): Function to use for collating instances to a batch.

        Returns:
            torch.utils.data.DataLoader: Dataloader over the given data, post processing.
        """

        # BEGIN CODE : trainer.make_dataloader

        # ADD YOUR CODE HERE
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle_data, collate_fn=collate_fn)
        # END CODE

    def train_step(self, x_batch, y_batch):
        """ Performs a step of training, on the training batch.

        Args:
            x_batch (torch.Tensor): Input batch.
            y_batch (torch.Tensor): Output batch.

        Returns:
            float: Training loss with the current model, on this batch.
        """

        # BEGIN CODE : trainer.train_step

        # ADD YOUR CODE HERE
        self.model.train()
        self.optimizer.zero_grad()
        x_batch = x_batch.cuda() if self.device == 'cuda' else x_batch
        y_batch = y_batch.cuda() if self.device == 'cuda' else y_batch
        y_pred = self.model.forward(x_batch)
        loss = self.criterion(y_pred, y_batch)
        loss.backward()
        self.optimizer.step()
        return float(loss.cpu().detach().numpy())
        # END CODE

    def eval_step(self, validation_dataloader):
        """ Perfoms an evaluation step, on the validation dataloader.

        Args:
            validation_dataloader (torch.utils.data.DataLoader): Dataloader for the validation dataset.

        Returns:
            float: Validation loss with the current model checkpoint.
        """

        # BEGIN CODE : trainer.eval_step

        # ADD YOUR CODE HERE
        total_loss = 0
        self.model.eval()
        for ind, (x, y) in enumerate(validation_dataloader):
            x = x.cuda() if self.device == 'cuda' else x
            y = y.cuda() if self.device == 'cuda' else y
            with torch.no_grad():
                y_pred = self.model.forward(x)
                total_loss += self.criterion(y_pred, y)
        val_loss = total_loss / (ind + 1)
        return float(val_loss.cpu().detach().numpy())
        # END CODE

    def cos_sim(self, x_batch, y_batch):
        x_batch = x_batch.to(self.device)
        y_batch = y_batch.to(self.device)
        self.model.eval()
        y_pred = self.model.forward(x_batch)
        return float((torch.mean(torch.sum(torch.nn.functional.normalize(y_pred, dim = 1)*torch.nn.functional.normalize(y_batch, dim = 1), dim = 1))).cpu().detach().numpy())

    def validation_cos_sim(self, val_dataloader):
        self.model.eval()
        total_loss = 0
        for ind, (x, y) in enumerate(val_dataloader):
            x = x.cuda() if self.device == 'cuda' else x
            y = y.cuda() if self.device == 'cuda' else y
            with torch.no_grad():
                total_loss += self.cos_sim(x, y)
        val_loss = total_loss / (ind + 1)
        return float(val_loss)


    def train(self, train_dataset, validation_dataset=None,
              num_epochs=10, batch_size=8, shuffle=True,
              save_steps=100, eval_steps=100, collate_fn=None, epoch_per_block = None):
        """ Handles the training loop for the model.

        Args:
            train_dataset (torch.utils.data.Dataset): Dataset to train on.
            validation_dataset (torch.utils.data.Dataset, optional): Data to validate on. Defaults to None.
            num_epochs (int, optional): Number of epochs to train for. Defaults to 10.
            batch_size (int, optional): Number of items to process per batch. Defaults to 8.
            shuffle (bool, optional): Whether to shuffle the data or not. Defaults to True.
            save_steps (int, optional): Number of steps post which a checkpoint should be saved. Defaults to 100.
            eval_steps (int, optional): Number of steps post which the model should be evaluated. Defaults to 100.
            collate_fn (function, optional): Function to use for collating instances to a batch.
        """

        current_checkpoint = 0
        self.model.to(self.device)
        self.model.train()

        with tqdm.tqdm(total = math.ceil(len(train_dataset) / batch_size) * num_epochs) as pbar:
            for epoch in range(num_epochs):
                if epoch_per_block != None and epoch % epoch_per_block == (epoch_per_block - 1):
                    self.model.add_block()
                train_dataloader      = self.make_dataloader(train_dataset, shuffle, batch_size, collate_fn)
                if validation_dataset is not None:
                    validation_dataloader = self.make_dataloader(validation_dataset, shuffle, batch_size, collate_fn)

                for batch, (x_batch, y_batch) in enumerate(train_dataloader):
                    pbar.set_description(f"Epoch {epoch+1} / {num_epochs}")

                    # If we are resuming training, skip this iteration
                    if current_checkpoint < self.last_checkpoint:
                        current_checkpoint += 1
                        pbar.update()
                        continue

                    # Do a step of training
                    loss = self.train_step(x_batch, y_batch)
                    self.loss_history['train'].append(loss)
                    pbar.set_postfix({ 'batch': batch+1, 'loss': loss })

                    current_checkpoint += 1
                    pbar.update()

                    # Evaluate after every eval_steps
                    if (current_checkpoint) % eval_steps == 0:
                        if validation_dataset is not None:
                            val_loss = self.eval_step(validation_dataloader)
                            val_sim = self.validation_cos_sim(validation_dataloader)
                            self.loss_history['valid'].append(val_loss)
                            self.loss_history['val_cosine_sim'].append(self.validation_cos_sim(validation_dataloader))
                            self.loss_history['train_cosine_sim'].append(self.cos_sim(x_batch, y_batch))
                        else:
                            val_loss = None

                        print('[>]', f"epoch #{epoch+1:{len(str(num_epochs))}},",
                              f"batch #{batch+1:{len(str(len(train_dataloader)))}}:",
                              "loss:", f"{loss:.8f}", '|', "val_loss:", f"{val_loss:.8f}","|",
                              "val cos sim:", f"{self.loss_history['val_cosine_sim'][-1]:.8f}","|",
                              "train cos sim:", f"{self.loss_history['train_cosine_sim'][-1]:.8f}")

                    # Save after every save_steps
                    if (current_checkpoint) % save_steps == 0:
                        self.save(current_checkpoint, { 'loss': loss, 'checkpoint': current_checkpoint })

                    # free unused resources
                    sync_vram()

            self.save(current_checkpoint)

    def resume(self):
        """ Resumes training session from the most recent checkpoint. """

        if checkpoints := os.listdir(self.directory):
            self.last_checkpoint = max(map(lambda x: int(x[11:]), filter(lambda x: 'checkpoint-' in x, checkpoints)))
            checkpoint_dir = os.path.join(self.directory, f"checkpoint-{self.last_checkpoint}")
            self.model.load_state_dict(torch.load(
                os.path.join(checkpoint_dir, "model.pt"),
                map_location=self.device
            ))
            self.model.to(self.device)
            self.optimizer.load_state_dict(torch.load(
                os.path.join(checkpoint_dir, "optimizer.pt"),
                map_location=self.device
            ))
            with open(os.path.join(checkpoint_dir, "loss.json"), 'r', encoding='utf-8') as ifile:
                self.loss_history = json.load(ifile)

    def save(self, checkpoint=None, metadata=None):
        """ Saves an associated model or a training checkpoint.

            If a checkpoint is specified, saves a checkpoint specific directory with optimizer data
                so that training can be resumed post that checkpoint.

        Args:
            checkpoint (int, optional): Checkpoint index. Defaults to None.
            metadata (dict[str, any], optional): Additional metadata to save alongside a checkpoint. Defaults to None.
        """

        if checkpoint is not None:
            checkpoint_dir = os.path.join(self.directory, f"checkpoint-{checkpoint}")
            os.makedirs(checkpoint_dir, exist_ok=True)
            torch.save(self.model.state_dict(), os.path.join(checkpoint_dir, "model.pt"))
            torch.save(self.optimizer.state_dict(), os.path.join(checkpoint_dir, "optimizer.pt"))
            with open(os.path.join(checkpoint_dir, "loss.json"), "w+", encoding='utf-8') as ofile:
                json.dump(self.loss_history, ofile, ensure_ascii=False, indent=2)
            if metadata:
                with open(os.path.join(checkpoint_dir, "metadata.json"), "w+", encoding='utf-8') as ofile:
                    json.dump(metadata, ofile, ensure_ascii=False, indent=2)
        else:
            torch.save(self.model, os.path.join(self.directory, "model.pt"))
            with open(os.path.join(self.directory, "loss.json"), "w+", encoding='utf-8') as ofile:
                json.dump(self.loss_history, ofile, ensure_ascii=False, indent=2)
            if metadata:
                with open(os.path.join(self.directory, "metadata.json"), "w+", encoding='utf-8') as ofile:
                    json.dump(metadata, ofile, ensure_ascii=False, indent=2)

## ==== END EVALUATION PORTION

## Embed model training

In [45]:
DIRECTORY_NAME = DIRECTORY
model_params = dict(
    inp_dim = INP_DIM,
    hid_dim = (INP_DIM + OUT_DIM)//2,
    out_dim = OUT_DIM,
    activation = 'relu',
    init_num_blocks = 3,
    layers_per_block = 3,
    dropout = 0.1
)

trainer_data_prams = dict(
    train_dataset=train_dataset,
    validation_dataset=val_dataset,
    collate_fn=None,
)

trainer_params = dict(
    num_epochs=10,
    batch_size=16,
    shuffle=True,
    save_steps=100,
    eval_steps=50,
    epoch_per_block = None
)

torch.manual_seed(42)
model = Embed_Adapter(**model_params)

optimizer = torch.optim.Adam(model.parameters(), lr = .01)
criterion = torch.nn.MSELoss()

trainer = Trainer(os.path.join(DIRECTORY_NAME, "adapter.dummy"),
    model, criterion, optimizer, )

In [ ]:
trainer.resume()

# Train as per specified training parameters.
trainer.train(**trainer_data_prams, **trainer_params)

Epoch 1 / 10:   0%|          | 1/15270 [00:00<2:35:11,  1.64it/s, batch=1, loss=1.12]


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


: 